# 🧠 Sephora — Activation Steering Research Notebook
> **The first local AI assistant with controllable internal behavior.**  
> Run this notebook on **Google Colab with a T4 GPU** (Runtime → Change runtime type → T4 GPU).

This notebook lets you:
- Load Mistral-7B locally in 4-bit quantization (~4 GB VRAM)
- Compute behavioral steering vectors from contrastive examples
- Run side-by-side steered vs unsteered comparisons
- Visualize token delta and latency benchmarks across all 5 presets

## ⚡ Step 0: Verify GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('❌ No GPU detected! Go to Runtime → Change runtime type → T4 GPU and restart.')
    raise SystemExit('GPU required.')

## 📦 Step 1: Install Dependencies

In [ ]:
# Install all required packages
!pip install -q transformers==4.44.0 accelerate bitsandbytes torch \
    huggingface_hub pyyaml python-dotenv loguru
print('✅ Dependencies installed.')

## 📁 Step 2: Clone Sephora Repository

In [ ]:
import os

if not os.path.exists('/content/sephora'):
    !git clone https://github.com/jonsnow273/sephora.git /content/sephora
    print('✅ Repository cloned.')
else:
    !git -C /content/sephora pull
    print('✅ Repository updated.')

os.chdir('/content/sephora')
import sys
sys.path.insert(0, '/content/sephora')
print(f'Working directory: {os.getcwd()}')

## 🔑 Step 3: HuggingFace Token (Optional but Recommended)
> Paste your free HF token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)  
> This gives you higher rate limits and faster downloads. Leave empty to skip.

In [ ]:
import os
from getpass import getpass

hf_token = getpass('Paste your HF token (or press Enter to skip): ').strip()
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print('✅ HF Token set.')
else:
    print('ℹ️  No token provided. Continuing unauthenticated.')

## 🤖 Step 4: Load Mistral-7B (4-bit Quantization)
> Downloads ~4 GB from HuggingFace. Takes ~2-3 minutes on first run.  
> Subsequent runs load from cache instantly.

In [ ]:
from core import config, logger
from llm import loader, engine

# Force Mistral 7B (already the default in configs/sephora_settings.yaml)
MODEL = 'mistralai/Mistral-7B-Instruct-v0.3'

print(f'Loading {MODEL} on {config.device} with {config.quantization} quantization...')
loader.load(model_name=MODEL)
print('✅ Model loaded!')

## 🎛️ Step 5: Initialize Activation Steering Engine

In [ ]:
from steering import SteeringEngine

se = SteeringEngine(loader.model, loader.tokenizer_wrapper, engine)
print('✅ Steering engine ready.')
print(f'Available presets: {list(se._direction_cache.keys()) or "(none calibrated yet)"}')

## 🔬 Step 6: Calibrate All Steering Vectors
> This runs contrastive prompt pairs through the model to compute  
> behavioral direction vectors for all 5 presets. Runs once and caches to disk.

In [ ]:
print('Calibrating all presets (this takes ~60 seconds)...')
se.calibrate()  # Calibrates: cautious, concise, detailed, creative
print(f'\n✅ Calibration complete!')
print(f'Cached directions: {list(se._direction_cache.keys())}')

## 💬 Step 7: Quick Single-Turn Chat

In [ ]:
YOUR_QUESTION = 'Explain what the internet is'  # ← Change this!

messages = [{'role': 'user', 'content': YOUR_QUESTION}]

se.reset()  # Neutral mode
response = engine.generate(messages)
print('=== Sephora (Neutral) ===')
print(response)

## ⚖️ Step 8: Side-by-Side Comparison (Core Research Feature)
> This is the main research contribution: same prompt, same model,  
> different internal behavior through activation steering.

In [ ]:
import time

PROMPT = 'Explain what the internet is'  # ← Change this!
PRESET = 'concise'   # ← Try: cautious, concise, detailed, creative
ALPHA  = 1.8         # ← Steering strength (0.0 to 2.8)

messages = [{'role': 'user', 'content': PROMPT}]
result = se.compare(messages, preset_name=PRESET, alpha=ALPHA)

m = result['metrics']
print(f'Prompt   : "{PROMPT}"')
print(f'Preset   : {PRESET} | Alpha: {ALPHA}')
print()
print(f'--- [1] BASELINE (Unsteered) | {m["unsteered_tokens"]} tokens | {m["unsteered_latency_ms"]}ms ---')
print(result['unsteered'])
print()
print(f'--- [2] STEERED: {PRESET} | {m["steered_tokens"]} tokens ({m["token_delta_percentage"]}%) | {m["steered_latency_ms"]}ms ---')
print(result['steered'])

## 📊 Step 9: Full Benchmark Across All Presets
> Runs your prompt through all 4 behavioral presets and prints a comparison table.  
> These numbers go directly into `docs/experiments.md`.

In [ ]:
BENCHMARK_PROMPT = 'How does memory management work in an operating system?'

print(f'Benchmark prompt: "{BENCHMARK_PROMPT}"\n')
print(f'{'Preset':<12} {'Tokens (steered)':<20} {'Token Delta %':<16} {'Latency (ms)'}')
print('-' * 64)

for preset_name in ['cautious', 'concise', 'detailed', 'creative']:
    msgs = [{'role': 'user', 'content': BENCHMARK_PROMPT}]
    res = se.compare(msgs, preset_name=preset_name)
    m = res['metrics']
    delta = f"{m['token_delta_percentage']}%"
    print(f'{preset_name:<12} {m["steered_tokens"]:<20} {delta:<16} {m["steered_latency_ms"]}')

print('\n✅ Benchmark complete. Copy these numbers into docs/experiments.md!')

## 🖥️ Step 10: Interactive Chat Loop

In [ ]:
# Run this cell to start a full interactive chat session in Colab
# Type /steer <preset> to change behavior, /compare <prompt> for side-by-side
# Type /quit to exit

import sys
sys.path.insert(0, '/content/sephora')
from main import run_interactive_cli

run_interactive_cli(preset_name='neutral', model_name='mistralai/Mistral-7B-Instruct-v0.3')